# Meta-evolution

A second loop watches the first and rewrites its search when it stalls.

*The search strategy is code, so it can be evolved like any other code.*

> Faithful to [EvoX](https://arxiv.org/abs/2602.23413) ([skydiscover](https://github.com/skydiscover-ai/skydiscover)): two coupled loops.

`similarity(a, b)` scores how alike two texts are. The seed compares characters, so it calls
"the door is **not** open" a near-match for "the door is open". Fixing that is not a tweak.

## 0. Setup

Two loops, two editable files — that split *is* the mechanism:

| path | rewritten by | EvoX's |
|---|---|---|
| `initial_program.py` | the **inner** loop — solutions | `initial_program.py` |
| `search_strategy.py` | the **outer** loop — the search itself | `EvolvedProgramDatabase` |
| `evaluator/` | nobody | `evaluator/` |

In [1]:
import os
import re
from dataclasses import dataclass, field
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

from evaluator.score import score   # protected — never enters a prompt

load_dotenv(find_dotenv(usecwd=True))
client = OpenAI(
    api_key=os.environ["DEEPINFRA_API_KEY"],
    base_url="https://api.deepinfra.com/v1/openai",
)
MODEL = "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"

PROGRAM  = Path("initial_program.py")    # the inner loop's target
STRATEGY = Path("search_strategy.py")    # the outer loop's target

W, TAU, GENERATIONS = 5, 0.01, 3   # window, stagnation threshold, outer rounds

## 1. The population

Every evaluated program is kept, not just the best — a strategy can only pick a parent that
still exists. `add` and `sample` are the two methods the outer loop is allowed to rewrite.

In [2]:
@dataclass
class Program:
    id: str
    solution: str
    metrics: dict = field(default_factory=dict)
    parent_id: str | None = None
    generation: int = 0


class ProgramDatabase:
    """The base. A strategy is a subclass that decides add() and sample()."""

    def __init__(self):
        self.programs: list[Program] = []

    def add(self, program: Program):
        raise NotImplementedError

    def sample(self, num_context_programs: int = 4):
        """-> (parent, context_programs). The whole search strategy is this method."""
        raise NotImplementedError

    def best(self) -> Program:
        return max(self.programs, key=lambda p: p.metrics["score"])


def load(src: str) -> type:
    """Compile a strategy. ProgramDatabase is injected — the file only names it."""
    ns = {"ProgramDatabase": ProgramDatabase, "Program": Program}
    exec(src, ns)
    return ns["EvolvedProgramDatabase"]


print(STRATEGY.read_text())

# EDITABLE — the OUTER loop rewrites this. EvoX's EvolvedProgramDatabase.


class EvolvedProgramDatabase(ProgramDatabase):   # noqa: F821 — injected by load()
    """The search strategy: which parent to mutate, and what to show alongside it.

    Greedy on purpose. Always the single best program, never any context — the
    fixed strategy EvoX exists to replace.
    """

    def add(self, program):
        self.programs.append(program)

    def sample(self, num_context_programs=4):
        parent = max(self.programs, key=lambda p: p.metrics["score"])
        return parent, []



## 2. The inner loop — evolve solutions

Sample a parent through the strategy, ask for one variation, score it, keep it. The score is a
correlation against held-out human judgement, so it is exact — nothing here is timed.

In [3]:
def propose(parent: Program, context: list[Program]) -> str | None:
    """One variation of one parent. Context is whatever the strategy chose to show."""
    shown = "".join(
        f"\nA sibling scoring {p.metrics['score']:.3f}:\n```python\n{p.solution}\n```\n"
        for p in context
    )
    msg = (
        "Improve `similarity(a, b) -> float`, which rates how alike two texts are, 0..1.\n"
        "It is scored by CORRELATION with human judgement on held-out pairs you cannot see.\n"
        "Humans call a paraphrase similar, a typo similar, and a negated sentence NOT similar.\n"
        "Pure stdlib. No imports beyond the standard library, no I/O, no network.\n\n"
        f"Current program, scoring {parent.metrics['score']:.3f}:\n"
        f"```python\n{parent.solution}\n```\n{shown}\n"
        "Propose ONE change. Reply with only the new `def similarity(a, b):` "
        "in a ```python block."
    )
    out = client.chat.completions.create(
        model=MODEL, max_tokens=900, messages=[{"role": "user", "content": msg}],
    ).choices[0].message.content
    m = re.search(r"```python\n(.*?)```", out, re.S)
    return m.group(1).strip() if m else None


def evolve(db, gen: int, log: list):
    """Phase I — W iterations under whatever strategy is loaded."""
    for _ in range(W):
        parent, context = db.sample()
        cand = propose(parent, context)
        s, note = score(cand) if cand else (None, "no code block")
        if s is not None:
            db.add(Program(id=f"g{gen}-{len(db.programs)}", solution=cand,
                           metrics={"score": s}, parent_id=parent.id, generation=gen))
        log.append((gen, s, note, parent.id, len(context)))
        print(f"    parent {parent.id:<7} +{len(context)} ctx  -> "
              f"{f'{s:.3f}' if s is not None else note[:34]}")

## 3. The outer loop — evolve the search

Watch a window of `W`, take `Δ = s_end - s_start`, and rewrite the strategy **only if**
`Δ < τ`. Demand-driven, not periodic: a search that is still climbing is left alone.

In [4]:
def rewrite_strategy(src: str, db, history: list[str]) -> str | None:
    """The meta-prompt. It sees the strategy and the population — never the evaluator."""
    pop = "\n".join(f"  {p.id:<7} score {p.metrics['score']:.3f}  parent {p.parent_id}"
                    for p in db.programs)
    msg = (
        "You tune the SEARCH, not the solution. An inner loop is evolving a program by "
        "repeatedly sampling a parent from a database and asking an LLM to vary it.\n"
        "It has stalled. Rewrite the database so the search escapes.\n\n"
        f"Current strategy:\n```python\n{src}\n```\n\n"
        f"Population so far:\n{pop}\n\n"
        f"Strategies already tried: {len(history)}\n\n"
        "Keep the class name `EvolvedProgramDatabase(ProgramDatabase)` and the signatures "
        "`add(self, program)` and `sample(self, num_context_programs=4)`. `sample` returns "
        "`(parent, context_programs)`. Programs have `.id`, `.solution`, `.metrics['score']`, "
        "`.parent_id`, `.generation`. `self.programs` is the list. Pure stdlib; `random` is "
        "allowed. Favour parent and context DIVERSITY over deterministic selection.\n"
        "Reply with only the class in a ```python block."
    )
    out = client.chat.completions.create(
        model=MODEL, max_tokens=900, messages=[{"role": "user", "content": msg}],
    ).choices[0].message.content
    m = re.search(r"```python\n(.*?)```", out, re.S)
    return m.group(1).strip() if m else None


def hot_swap(new_src: str, db):
    """Load the new strategy and carry the population over. Any failure -> keep the old one."""
    fresh = load(new_src)()               # raises if the class is malformed
    for p in db.programs:
        fresh.add(p)
    fresh.sample()                        # raises if sample() is broken
    return fresh

## 4. Run

`git diff` afterwards to see both files the loops rewrote.

In [5]:
seed_src = PROGRAM.read_text()
seed_score, note = score(seed_src)
print(f"seed similarity()  corr {seed_score:.3f}  {note}\n")

strategy_src, history, log = STRATEGY.read_text(), [], []
db = load(strategy_src)()
db.add(Program(id="seed", solution=seed_src, metrics={"score": seed_score}))

for gen in range(1, GENERATIONS + 1):
    s_start = db.best().metrics["score"]
    print(f"gen {gen}  strategy v{len(history) + 1}  best {s_start:.3f}")
    evolve(db, gen, log)                                     # Phase I

    delta = db.best().metrics["score"] - s_start             # Phase II
    print(f"  delta {delta:+.3f} over {W}", end="  ")

    if delta < TAU and gen < GENERATIONS:                    # Phase III — only if stalled
        print("-> STALLED, rewriting the search")
        cand = rewrite_strategy(strategy_src, db, history)
        try:
            db = hot_swap(cand, db)
            history.append(strategy_src)
            strategy_src = cand
            STRATEGY.write_text(cand + "\n")
        except Exception as e:            # fallback/restore — a broken strategy is discarded
            print(f"       new strategy rejected ({type(e).__name__}), keeping the old one")
    else:
        print("-> still climbing, strategy untouched" if delta >= TAU else "")

best = db.best()
PROGRAM.write_text(best.solution + "\n")
print(f"\nseed {seed_score:.3f} -> best {best.metrics['score']:.3f}   "
      f"({len(db.programs)} programs, {len(history) + 1} strategies)")

seed similarity()  corr 0.335  ok

gen 1  strategy v1  best 0.335


    parent seed    +0 ctx  -> 0.239


    parent seed    +0 ctx  -> 0.183


    parent seed    +0 ctx  -> 0.335


    parent seed    +0 ctx  -> 0.335


...

    parent g1-3    +4 ctx  -> 0.376


    parent g1-3    +4 ctx  -> 0.264


    parent g2-7    +4 ctx  -> 0.371


    parent seed    +4 ctx  -> 0.471


    parent g1-3    +4 ctx  -> 0.479
  delta +0.144 over 5  -> still climbing, strategy untouched

seed 0.335 -> best 0.479   (15 programs, 2 strategies)


## 5. What the outer loop wrote

The point of the mechanism: the search strategy is code, and a loop rewrote it.

In [6]:
for i, src in enumerate(history + [strategy_src], 1):
    print(f"{'='*70}\nstrategy v{i}\n{'='*70}\n{src}\n")

print(f"{'='*70}\nbest similarity()  corr {best.metrics['score']:.3f}\n{'='*70}")
print(best.solution)

strategy v1
# EDITABLE — the OUTER loop rewrites this. EvoX's EvolvedProgramDatabase.


class EvolvedProgramDatabase(ProgramDatabase):   # noqa: F821 — injected by load()
    """The search strategy: which parent to mutate, and what to show alongside it.

    Greedy on purpose. Always the single best program, never any context — the
    fixed strategy EvoX exists to replace.
    """

    def add(self, program):
        self.programs.append(program)

    def sample(self, num_context_programs=4):
        parent = max(self.programs, key=lambda p: p.metrics["score"])
        return parent, []


strategy v2
import random

class EvolvedProgramDatabase(ProgramDatabase):
    """The search strategy: which parent to mutate, and what to show alongside it.

    Fosters diversity by sampling parents and context programs from different generations and with varying scores.
    """

    def add(self, program):
        self.programs.append(program)

    def sample(self, num_context_programs=4):
        

## Key findings

**The greedy strategy collapsed onto one parent.** All five of gen 1's proposals mutated `seed` —
`sample()` returns the argmax, nothing beat `0.335`, so the search asked the same question five
times over. `Δ = 0.000`. That is the fixed strategy EvoX aims at, failing on cue.

**The rewrite changed the question, not the answer.** v2 buckets the population by generation,
draws its parent from a random bucket, and shows four programs from *other* buckets as context —
20 lines of new search policy, written by a loop and hot-swapped mid-run (§5). The inner loop
then found the seed's blind spot on its own: the best program penalises a mismatched negation.

**One trace shows the mechanism, not the gain.** v2 stalled too, in gen 2; the `+0.144` arrived in
gen 3 under that same v2. So this run shows the rewrite happening and the search escaping — not
that the rewrite caused the escape. Both safeguards fired unprompted: a candidate returned `-0.5`
and lost on contract, and the gen-2 rewrite raised `NameError` and was restored. One generation
lost each, not the run.

## Weaknesses

| Weakness | What happens | Fix |
|---|---|---|
| **One run is an anecdote** | Proposals are sampled. v2 stalled in gen 2 and climbed `+0.144` in gen 3 — same strategy, so nothing here separates the rewrite from luck | Run fixed-vs-evolved N times and report the spread, not one trace |
| **The meta loop is a model too** | Its second rewrite didn't compile — `NameError`, 1 of 2 attempts. It fails at the same rate as the inner loop, and costs a whole window when it does | `hot_swap` already restores; budget for the waste |
| **Stagnation is a threshold, not a diagnosis** | `Δ < τ` fires on any flat window. A search that is merely slow looks identical to one that is stuck, and gen 1 stalled because the strategy was greedy — the trigger can't tell you that | Compare `Δ` against the window's own variance |
| **`W` and `τ` are hand-picked** | `W=5, τ=0.01` against a 24-pair evaluator. EvoX sizes `W` at 10% of budget; here it is a guess that decides how fast the outer loop is even allowed to react | Calibrate against a run that never rewrites |
| **Selection can't invent** | `sample()` only chooses among programs that exist. No selection rule finds an idea the population never contained — the variation operator is the other half | EvoX evolves that half too; this module does not |
| **The boundary is convention** | `exec` runs model-written solutions *and* model-written strategies in-process | Subprocess, timeout |
| **24 pairs, hand-written** | A correlation over 24 pairs I wrote myself is easy to overfit and is not human judgement in any real sense | More pairs, a held-out split, real annotations |